# Custom Adventurers

Create your own adventurer by subclassing `BaseAdventurer`.

Two patterns are supported:
- **Class variable** `system_prompt` (simplest)
- **`@property`** `system_prompt` (dynamic prompts)

In [ ]:
from dotenv import load_dotenv

load_dotenv()

## Option 1: Class variable prompt

In [ ]:
from guildmaster_ai import BaseAdventurer, GuildBuilder


class HaikuAdventurer(BaseAdventurer):
    """Responds only in haiku."""

    system_prompt = (
        "You are a haiku poet. Every response must be exactly one haiku "
        "(5-7-5 syllables). Nothing else."
    )


guild = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(HaikuAdventurer(name="Basho"))
    .build()
)

result = await guild.post_quest("Tell me about the ocean.")
print(result.summary)

## Option 2: Dynamic prompt via `@property`

In [ ]:
class PersonaAdventurer(BaseAdventurer):
    """Takes on any persona based on its name."""

    @property
    def system_prompt(self) -> str:
        return (
            f"You are {self.name}. Stay in character at all times. "
            "Keep responses concise (2-3 sentences)."
        )


guild2 = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(PersonaAdventurer(name="a pirate captain"))
    .build()
)

result = await guild2.post_quest("What is a REST API?")
print(result.summary)

## Custom `execute()` method

Override `execute()` when you need custom logic beyond the default tool-calling loop.

In [ ]:
from guildmaster_ai.core.messages import QuestResult
from guildmaster_ai.core.quest import Quest


class SummaryAdventurer(BaseAdventurer):
    """Always appends a TL;DR line to its response."""

    system_prompt = "You are a helpful assistant. Be thorough but concise."

    async def execute(self, quest: Quest) -> QuestResult:
        # Use the built-in conversation helpers
        self._reset_conversation()
        self._add_user_message(quest.description)
        response = await self._call_llm()

        # Custom post-processing
        summary = response.content + "\n\nTL;DR: Quest complete."

        return QuestResult(
            sender=self.name or self.id,
            quest_id=quest.id,
            success=True,
            summary=summary,
        )


guild3 = (
    GuildBuilder()
    .with_llm_provider("openrouter")
    .register_adventurer(SummaryAdventurer(name="Summarizer"))
    .build()
)

result = await guild3.post_quest("What is Python's GIL?")
print(result.summary)